# 📘 Fincept Notebook — Yield Curve & Recession Signal

**Economics · Hard · ~30 min · pandas + numpy**

The shape of the U.S. Treasury yield curve is one of the most-watched macro indicators. When short-term yields rise above long-term yields (an *inversion*), it has historically preceded recessions. In this notebook we build curves for a normal day and an inverted day, interpolate yields at any maturity, and compute the classic recession spreads.

**What you'll learn**
- Build a pandas DataFrame of Treasury yields across tenors (3M to 30Y)
- Interpolate yields at off-grid maturities with `np.interp`
- Compute the 10y-2y and 10y-3m spreads and classify the curve shape
- Explain why a negative 10y-2y spread is read as a recession signal

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. The yield-curve data

We embed two real-world-shaped snapshots of constant-maturity Treasury (CMT) yields, quoted in percent. The **2021-06-30** curve is upward-sloping (*normal*): you get paid more to lend for longer. The **2023-07-03** curve is *inverted*: the 3M bill out-yields the 10Y note — a textbook recession warning.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

# Tenors in years, and CMT yields (%) for two dates.
TENORS = {"3M": 0.25, "6M": 0.5, "1Y": 1, "2Y": 2, "3Y": 3,
          "5Y": 5, "7Y": 7, "10Y": 10, "20Y": 20, "30Y": 30}
YIELDS = {
    "2021-06-30": [0.05, 0.06, 0.07, 0.25, 0.46, 0.87, 1.21, 1.45, 2.00, 2.06],  # normal
    "2023-07-03": [5.43, 5.47, 5.40, 4.94, 4.66, 4.36, 4.23, 3.86, 4.16, 3.91],  # inverted
}

df = pd.DataFrame(YIELDS, index=pd.Index(TENORS.keys(), name="Tenor"))
df.insert(0, "Maturity(yr)", list(TENORS.values()))

print("U.S. Treasury constant-maturity yields (%)")
print("=" * 48)
print(df.to_string())


## 2. Interpolating yields at any maturity

The market only quotes a handful of tenors, but you often need a yield at an *off-grid* point — say a bond with 4 years left, or an 18-month liability. Because the curve is smooth, simple **linear interpolation** between the two neighbouring quotes is a reasonable first approximation. `np.interp(x, xp, fp)` does exactly this: given the grid maturities `xp` and their yields `fp`, it returns the yield at each requested maturity `x`.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

TENORS = {"3M": 0.25, "6M": 0.5, "1Y": 1, "2Y": 2, "3Y": 3,
          "5Y": 5, "7Y": 7, "10Y": 10, "20Y": 20, "30Y": 30}
YIELDS = {
    "2021-06-30": [0.05, 0.06, 0.07, 0.25, 0.46, 0.87, 1.21, 1.45, 2.00, 2.06],
    "2023-07-03": [5.43, 5.47, 5.40, 4.94, 4.66, 4.36, 4.23, 3.86, 4.16, 3.91],
}

xp = np.array(list(TENORS.values()))  # grid maturities in years
targets = [0.75, 1.5, 4.0, 8.0, 15.0, 25.0]  # off-grid maturities we want

rows = []
for date, ys in YIELDS.items():
    fp = np.array(ys)
    interp = np.interp(targets, xp, fp)
    for m, y in zip(targets, interp):
        rows.append({"Date": date, "Maturity(yr)": m, "Interp yield (%)": round(float(y), 3)})

out = pd.DataFrame(rows).pivot(index="Maturity(yr)", columns="Date", values="Interp yield (%)")
print("Linearly interpolated yields at off-grid maturities")
print("=" * 52)
print(out.to_string())


## 3. The recession spreads & curve shape

Two spreads matter most:

- **10y − 2y**: the classic slope. It has gone negative before *every* U.S. recession since the 1970s.
- **10y − 3m**: favoured by the Fed's own research as the cleanest predictor.

We classify the overall shape from the 10y−2y spread: clearly positive → **normal**, near zero → **flat**, negative → **inverted**.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

TENORS = ["3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
YIELDS = {
    "2021-06-30": [0.05, 0.06, 0.07, 0.25, 0.46, 0.87, 1.21, 1.45, 2.00, 2.06],
    "2023-07-03": [5.43, 5.47, 5.40, 4.94, 4.66, 4.36, 4.23, 3.86, 4.16, 3.91],
}
df = pd.DataFrame(YIELDS, index=TENORS)

def classify(spread_10_2):
    if spread_10_2 < -0.10:
        return "INVERTED"
    if spread_10_2 <= 0.25:
        return "FLAT"
    return "NORMAL"

rows = []
for date in df.columns:
    s = df[date]
    sp_10_2 = s["10Y"] - s["2Y"]
    sp_10_3m = s["10Y"] - s["3M"]
    rows.append({
        "Date": date,
        "2Y (%)": s["2Y"], "10Y (%)": s["10Y"],
        "10y-2y (bp)": round(sp_10_2 * 100, 1),
        "10y-3m (bp)": round(sp_10_3m * 100, 1),
        "Shape": classify(sp_10_2),
        "Recession signal": "YES" if sp_10_2 < 0 else "no",
    })

summary = pd.DataFrame(rows).set_index("Date")
print("Yield-curve spreads and classification")
print("=" * 64)
print(summary.to_string())


## 4. Reading the signal

Compare the two dates side by side. On the normal day the slope is steeply positive (the 10Y yields ~120bp more than the 2Y); on the inverted day the 2Y *out-yields* the 10Y by over 100bp. We print a tenor-by-tenor difference so you can see where the inversion bites hardest.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

TENORS = ["3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
MAT = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
YIELDS = {
    "2021-06-30": [0.05, 0.06, 0.07, 0.25, 0.46, 0.87, 1.21, 1.45, 2.00, 2.06],
    "2023-07-03": [5.43, 5.47, 5.40, 4.94, 4.66, 4.36, 4.23, 3.86, 4.16, 3.91],
}
df = pd.DataFrame(YIELDS, index=TENORS)
df["Change (bp)"] = ((df["2023-07-03"] - df["2021-06-30"]) * 100).round(0).astype(int)

print("Normal vs inverted curve, tenor by tenor")
print("=" * 56)
print(df.to_string())
print()

# Slope of each curve: yield(30Y) - yield(3M)
for date in ["2021-06-30", "2023-07-03"]:
    slope = df.loc["30Y", date] - df.loc["3M", date]
    direction = "upward (normal)" if slope > 0 else "downward (inverted)"
    print(f"{date}: 30Y-3M slope = {slope:+.2f} pts -> {direction}")

print()
print("Takeaway: a negative 10y-2y spread means investors expect the Fed to")
print("cut rates in the future (weaker growth), so they lock in long yields now —")
print("the curve inverts. Historically this has led recessions by ~6-18 months.")


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
